# Scaling the inputs of a neural network
Age (tens) and salary (tens of thousands) on very different scales: the same network learns nothing on the raw
columns and over 90% accuracy on standardized ones. Results go to `data/` for the figures in `images/`.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
import numpy as np
import pandas as pd
import keras
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

tf.config.experimental.enable_op_determinism()   # same numbers on every run

## The data
400 users of a social network: age, estimated salary, and whether they bought the advertised product.

In [ ]:
df = pd.read_csv("data/Social_Network_Ads.csv")
df = df[["Age", "EstimatedSalary", "Purchased"]]     # drop User ID and Gender
print(df.describe().round(0))

In [ ]:
X = df[["Age", "EstimatedSalary"]].values
y = df["Purchased"].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

## The network
One hidden layer of 128 ReLU nodes and a sigmoid output node.

In [ ]:
def build(seed=0):
    keras.utils.set_random_seed(seed)          # same starting weights for both runs
    model = keras.Sequential([
        keras.Input(shape=(2,)),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

build().summary()

## 1. Training on the raw columns

In [ ]:
model_raw = build()
h_raw = model_raw.fit(X_train, y_train, epochs=100, validation_data=(X_test, y_test), verbose=0).history
va = np.array(h_raw["val_accuracy"])
print("validation accuracy, last 10 epochs:", va[-10:].round(2))
print(f"range over all epochs: {va.min():.2f} to {va.max():.2f}")

## 2. Training on standardized columns

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)    # learn mean and std on the training rows only
X_test_scaled = scaler.transform(X_test)
print(pd.DataFrame(X_train_scaled, columns=["Age", "EstimatedSalary"]).describe().round(2))

In [ ]:
model_scaled = build()
h_scaled = model_scaled.fit(X_train_scaled, y_train, epochs=100,
                            validation_data=(X_test_scaled, y_test), verbose=0).history
print("validation accuracy, last 10 epochs:", np.array(h_scaled["val_accuracy"])[-10:].round(2))

## 3. Why: the gradient of each input's weights
At the start of training, the gradient of a first-layer weight is proportional to its input. We compare the
average size of the gradients of the age weights and the salary weights.

In [ ]:
def first_layer_gradients(model, X_in):
    with tf.GradientTape() as tape:
        p = model(X_in.astype("float32"))
        loss = tf.reduce_mean(keras.losses.binary_crossentropy(y_train.reshape(-1, 1).astype("float32"), p))
    g = tape.gradient(loss, model.layers[0].kernel).numpy()     # shape (2, 128): one row per input
    return np.abs(g).mean(axis=1)

rows = []
for name, Xi in [("raw", X_train), ("standardized", X_train_scaled)]:
    g_age, g_sal = first_layer_gradients(build(), Xi)
    rows.append({"inputs": name, "age": g_age, "salary": g_sal})
    print(f"{name:13s} mean |grad| age weights {g_age:.3g}, salary weights {g_sal:.3g}, ratio {g_sal / g_age:.2f}")

## Saving results for the figures

In [ ]:
pd.DataFrame({"epoch": np.arange(1, 101),
              "raw_val_accuracy": h_raw["val_accuracy"], "scaled_val_accuracy": h_scaled["val_accuracy"],
              "raw_loss": h_raw["loss"], "scaled_loss": h_scaled["loss"]}).round(4).to_csv(
    "data/histories.csv", index=False)
pd.DataFrame(rows).to_csv("data/gradients.csv", index=False)
print("saved")